# Faza deweloperska - wyniki (E1-E5)

Czyta gotowe katalogi przebiegów z `results/runs/` i pliki adnotacji, i składa z nich tabele wszystkich rozstrzygnięć fazy deweloperskiej. Nie liczy nic na nagraniach i nie ładuje żadnego modelu, więc wykonuje się w sekundy i można go uruchamiać po każdym kolejnym przebiegu. Bloki bez przebiegu nie wywalają się, tylko wypisują, czego brakuje.

**Wymaga:** przebiegów części deweloperskiej, kolejno (szczegóły w `docs/04_instrukcja_dev.md`):

```powershell
python scripts/run_experiment.py configs/e1a_office.yaml --split dev    # oraz e1b, e1c
python scripts/run_experiment.py configs/e2a_office.yaml --split dev    # oraz e2ap, e2b
python scripts/run_features.py   configs/e3b_office.yaml --split dev    # ekstrakcja cech E3-E5
python scripts/run_experiment.py configs/e3b_office.yaml --split dev    # oraz e3c, e4b, e4c, e5b, e5c
```

Dla obu seriali naraz to samo bez sufiksu `_office`. Rozstrzygnięcie wymaga obu: reguła z rozdziału 4 patrzy na zgodność kierunku w każdym serialu i na przedział ufności na ich połączonych odcinkach.

Wartości Recall@K są w procentach, różnice w punktach procentowych, separatorem dziesiętnym jest przecinek. Nawias kwadratowy to 95% przedział ufności różnicy; liczba bez przedziału (`n=...`) oznacza, że jednostek wnioskowania było za mało, żeby go policzyć.

**Zapisuje:** nic, tylko wypisuje.

**Dalej:** wpisanie werdyktów do `configs/frozen.yaml` (ręcznie, etap drugi: `segmentation`, `scene_embedding`, `caption`, `objects`, `face_regions`, `motion`), a potem `python scripts/make_configs.py --execute`, które wypisze z nich 57 konfiguracji pochodnych. `frozen.yaml` jest jedynym miejscem, w którym decyzja zapada.

In [ ]:
import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.annotation import registry as reg
from src.data import datasets, queries
from src.evaluation import compare, relevance, tables
from src.evaluation import runs as runs_module
from src.segmentation import segments as seg
from src.utils import experiments as exp

for module in (datasets, queries, compare, relevance, runs_module, tables, reg, seg, exp):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory
load_queries = queries.load_queries
relevance_sets = relevance.relevance_sets

SPLIT = "dev"
METRIC = "recall@10"

# Which label means what, which variants belong to which experiment, how a
# dataset is spelled: all of it lives in src/utils/experiments.py, so a table
# here and a generated configuration cannot disagree. Bound to local names
# because every block below reads them.
DATASETS = list(exp.SERIES)       # the development phase runs on the two series
LABEL = exp.DATASET_NAMES
STRATEGIES = exp.STRATEGY_NAMES

E1, REFERENCE_E1 = list(exp.E1), exp.E1_REFERENCE
E2, REFERENCE_E2 = list(exp.E2), exp.E2_REFERENCE
E2_DECISION = list(exp.E2_DECISION)        # the control variants do not take part
E2_NAMES = {label: exp.display(label) for label in exp.E2}

# (the simpler variant first - it is the reference), label of the pair
PAIRS = [tuple(pair) for pair in exp.PAIRS]

# one loader for the whole repository (src/evaluation/runs.py): the newest
# directory of a (label, dataset) wins, and nothing here reimplements that
runs = runs_module.load_runs(SPLIT)


def available(labels):
    """Runs of the given labels, on the datasets ALL of them cover."""
    present, shared, skipped = runs_module.available(runs, labels)
    for label, reason in skipped:
        print(f"  {label}: {reason}")
    return runs_module.select(runs, present, shared), shared



def pair(simpler, complex_):
    """Decision for one E3-E5 pair; returns ``(result, verdict)`` or ``(None, None)``.

    The SIMPLER variant is the reference and the complex one is the candidate, so
    the difference is "complex minus simpler" and a positive value means the
    complex variant won. When the interval covers zero nothing qualifies and
    :func:`compare.decide` falls back to the first label of the simplicity order,
    which is the simpler variant -- the rule of chapter 4. Passing the pair the
    other way round inverts that fallback, which is why this lives in one place.
    """
    group, _ = available([simpler, complex_])
    if len(group) < 2:
        return None, None
    result = compare.compare_variant(group[complex_], group[simpler], METRIC)
    return result, compare.decide({complex_: result}, [simpler, complex_])


def missing(labels):
    """Prints what has to be run for a block to have anything to show."""
    gaps = [label for label in labels if label not in runs]
    if gaps:
        print(f"no runs for: {', '.join(gaps)} - see the header of the notebook")
    return bool(gaps)


print(f"runs found for split {SPLIT!r}:")
for label in sorted(runs):
    print(f"  {label:<8}{', '.join(sorted(runs[label]))}")
if not runs:
    print("  (none yet)")

## 1. Zapytania

Liczby biorą się z rejestru zbiorczego: ile adnotacji wpadło na wejściu, ile odrzucono, ile poprawiono i ile zostało.

In [ ]:
rows = []
for dataset in DATASETS:
    path = ROOT / "data" / "interim" / datasets.dataset_dir(dataset) /         f"{datasets.dataset_dir(dataset)}_annotations.csv"
    if not path.exists():
        rows.append([LABEL[dataset]] + ["-"] * 6)
        continue
    register = reg.load(path)
    group = [r for r in register if r["split"] == SPLIT]
    accepted = [r for r in group if r["status"] == reg.STATUS_ACCEPTED]
    rows.append([LABEL[dataset], len({r["episode"] for r in group}), len(group),
                 len(group) - len(accepted),
                 sum(1 for r in accepted if r["edited_duration"] == reg.YES),
                 sum(1 for r in accepted if r["edited_desc"] == reg.YES),
                 len(accepted)])

tables.show("Zapytania czesci deweloperskiej",
            ["Zbior", "Odcinkow", "Poczatkowych", "Odrzuconych",
             "Popr. czasu", "Popr. tresci", "Koncowych"], rows,
            note="Poprawki liczone wsrod przyjetych; jedna adnotacja moze miec obie. "
                 "Samo przesuniecie w czasie nie jest poprawka - liczy sie zmiana "
                 "dlugosci (edited_duration) albo tresci (edited_desc).")

## 2. Statystyka segmentacji

Liczba fragmentów, ich średnia długość mierzona na osi treści oraz udział zdarzeń zajmujących więcej niż jeden fragment. Ten ostatni liczony jest regułą relewancji z rozdziału 4, czyli tak samo, jak liczą go przebiegi.

In [ ]:
rows = []
for dataset in DATASETS:
    try:
        queries = load_queries(dataset, SPLIT)
    except FileNotFoundError:
        queries = []
    for strategy, name in STRATEGIES.items():
        path = datasets.segments_csv(strategy, dataset)
        fragments = [r for r in seg.load(path) if r["split"] == SPLIT] \
            if path.exists() else []
        if not fragments:
            rows.append([LABEL[dataset], name, "-", "-", "-"])
            continue
        lengths = [r["duration"] for r in fragments]
        relevance = relevance_sets(queries, fragments) if queries else {}
        spanning = [len(v) for v in relevance.values() if v]
        share = (100 * sum(1 for n in spanning if n > 1) / len(spanning)
                 if spanning else None)
        rows.append([LABEL[dataset], name, len(fragments),
                     tables.number(sum(lengths) / len(lengths), 1),
                     tables.number(share, 1) if share is not None else "-"])

tables.show("Segmentacja czesci deweloperskiej",
            ["Zbior", "Strategia podzialu", "Fragmentow", "Sr. dlugosc [s]",
             "Zdarzen wielofragm. [%]"], rows,
            note="Dlugosc jest mierzona na osi tresci: sekundy maski przejscia sie nie licza.")

## 3. E1 - wybór strategii segmentacji

Recall@10 każdego wariantu i jego różnica względem stałych okien (E1-A), z 95% przedziałem liczonym po odcinkach-klastrach. Wiersz "polaczone" to przedział na połączonych odcinkach obu seriali, ten, którego dotyczy reguła decyzji.

In [ ]:
group, columns = available(E1)
if not missing(E1) and group:
    reference = group[REFERENCE_E1]
    results = {label: compare.compare_variant(runs_of, reference, METRIC)
               for label, runs_of in group.items() if label != REFERENCE_E1}

    header = ["Konfiguracja"]
    for dataset in columns:
        header += [f"{LABEL[dataset]} R@10", f"{LABEL[dataset]} delta [95% PU]"]
    header += ["Polaczone delta [95% PU]"]

    rows = []
    for label in E1:
        if label not in group:
            continue
        cells = [f"{label}: {STRATEGIES[list(STRATEGIES)[E1.index(label)]]}"]
        for dataset in columns:
            if label == REFERENCE_E1:
                value = compare.query_mean(reference[dataset]["per_query"], METRIC)
                cells += [tables.percent(value), "-"]
            else:
                entry = results[label]["per_dataset"][dataset]
                cells += [tables.percent(entry["value"]), tables.interval(entry)]
        cells += ["-" if label == REFERENCE_E1
                  else tables.interval(results[label]["pooled"])]
        rows.append(cells)

    tables.show("E1: strategie segmentacji", header, rows)

    verdict = compare.decide(results, [l for l in E1 if l in group])
    print(f"\nwerdykt: {verdict['winner']} - {verdict['reason']}")
    if verdict["provisional"]:
        print(f"UWAGA: przedzial polaczony opiera sie na {verdict['series']} serialu;"
              " regula wymaga obu")

## 4. E1 - kontrast konfirmacyjny `wymaga_scenerii`

Ta sama różnica względem E1-A, ale osobno na zapytaniach ze znacznikiem i na pozostałych, plus różnica tych dwóch różnic z przedziałem. To ona rozstrzyga, czy przewaga strategii opartych na ujęciach koncentruje się na zapytaniach o scenerię. Przy nieuzupełnionych znacznikach blok to wypisze zamiast liczyć.

In [ ]:
CONTRAST_E1 = "requirements:wymaga_scenerii"

group, columns = available(E1)
if missing(E1) or not group:
    pass
else:
    matching, rest, empty = {}, {}, []
    for dataset in columns:
        matching[dataset], rest[dataset] = compare.subset_ids(dataset, SPLIT, CONTRAST_E1)
        if not matching[dataset]:
            empty.append(dataset)
        else:
            print(f"{LABEL[dataset]}: {len(matching[dataset])} zapytan ze znacznikiem,"
                  f" {len(rest[dataset])} bez")
    if empty:
        print(f"brak znacznikow w: {', '.join(LABEL[d] for d in empty)}"
              " - uzupelnij <zbior>_query_tags.csv")
    else:
        reference = group[REFERENCE_E1]
        rows = []
        for label in E1:
            if label == REFERENCE_E1 or label not in group:
                continue
            contrast = compare.contrast_variant(group[label], reference, METRIC,
                                                matching, rest)
            for dataset in columns:
                entry = contrast["per_dataset"][dataset]
                rows.append([f"{label} / {LABEL[dataset]}",
                             tables.points(entry["matching_delta"]),
                             tables.points(entry["rest_delta"]),
                             tables.interval(entry)])
            rows.append([f"{label} / polaczone", "-", "-",
                         tables.interval(contrast["pooled"])])

        tables.show("E1: kontrast wymaga_scenerii",
                    ["Konfiguracja", "delta ze znacznikiem", "delta bez znacznika",
                     "Roznica [95% PU]"], rows)

## 5. E1 - próg dopasowany do liczebności kolekcji

Strategie E1 dzielą ten sam materiał na różną liczbę fragmentów, więc przy stałym progu $K$ każda pokazuje inny odsetek rankingu - ta wytwarzająca fragmenty krótsze i liczniejsze jest oceniana surowiej niezależnie od jakości podziału. Próg dopasowany $K_i = \max(1, \mathrm{round}(K_0 \cdot |\mathcal{F}_i| / |\mathcal{F}_0|))$, $K_0 = 10$ dla wariantu odniesienia, zrównuje ten odsetek.

$R@K_i$ jest analizą wrażliwości: nie uczestniczy w rozstrzyganiu i nie ma przedziałów ufności. Pokazuje tylko, czy wniosek z miary głównej nie jest artefaktem różnicy w liczbie fragmentów.

In [ ]:
K0 = 10       # threshold of the main metric, set for the reference variant


def recall_at(per_query, k):
    """R@k from the rank of the first relevant fragment, for any k."""
    hits = sum(1 for q in per_query if q["rank"] is not None and q["rank"] <= k)
    return hits / len(per_query) if per_query else 0.0


group, columns = available(E1)
if not missing(E1) and group:
    rows = []
    for dataset in columns:
        rows.append(LABEL[dataset])                    # a plain string becomes a subheading
        size0 = group[REFERENCE_E1][dataset]["metrics"]["collection_size"]
        for label in E1:
            if label not in group:
                continue
            run = group[label][dataset]
            size = run["metrics"]["collection_size"]
            k = max(1, round(K0 * size / size0))
            per_query = run["per_query"]
            rows.append([f"{label}: {STRATEGIES[list(STRATEGIES)[E1.index(label)]]}",
                         size, k,
                         tables.percent(recall_at(per_query, 1)),
                         tables.percent(recall_at(per_query, 5)),
                         tables.percent(recall_at(per_query, 10)),
                         tables.percent(recall_at(per_query, k))])

    tables.show("E1: prog dopasowany do liczebnosci kolekcji",
                ["Konfiguracja", "|F_i|", "K_i", "R@1", "R@5", "R@10", "R@K_i"], rows,
                note="R@K_i to analiza wrazliwosci - bez przedzialow ufnosci i bez "
                     "udzialu w decyzji. Dla wariantu odniesienia K_i = K_0 = 10, "
                     "wiec R@K_i pokrywa sie z R@10.")

## 6. Kontrola progu - miary poboczne

Wnioskowanie prowadzone jest wyłącznie dla miary głównej (`recall@10`): tylko dla niej liczone są przedziały ufności i tylko na niej opiera się werdykt. Miary poboczne pokazują, czy wniosek nie zależy od wyboru progu, i podawane są bez przedziałów i bez orzekania o istotności - porównania prowadzone równolegle na kilku miarach zwiększałyby ryzyko, że zaobserwowana różnica okaże się pozorna.

Wartości w tabeli to poziomy miary, czyli średnie po zapytaniach. Różnica w nawiasie jest różnicą między konfiguracjami, więc liczy się ją po odcinkach; nie jest różnicą dwóch wypisanych poziomów i nie powinna być tak odczytywana.

Malejąca różnica przy pogłębianiu listy nie zmienia rozstrzygnięcia ani nie uzasadnia zmiany miary głównej po zobaczeniu wyników; opisuje się to jako wpływ głębokości przeglądanej listy na wielkość efektu.

In [ ]:
BLOCKS = {"E1": (E1, REFERENCE_E1), "E2": (E2, REFERENCE_E2)}

for name, (labels, reference_label) in BLOCKS.items():
    group, columns = available(labels)
    if not group or reference_label not in group:
        missing(labels)
        continue
    reference = group[reference_label]
    header = ["Konfiguracja", "Zbior"] + list(compare.SECONDARY_METRICS)

    rows = []
    for label in labels:
        if label not in group:
            continue
        # the reference is compared with itself, so its own values appear too and
        # the table reads without looking anything up elsewhere
        sizes = compare.effect_sizes(group[label], reference)
        for dataset in columns:
            entry = sizes[dataset]
            if label == reference_label:
                cells = [tables.percent(entry[m]["value"])
                         for m in compare.SECONDARY_METRICS]
            else:
                cells = [f"{tables.percent(entry[m]['value'])}"
                         f"  ({tables.points(entry[m]['delta'])})"
                         for m in compare.SECONDARY_METRICS]
            rows.append([label, LABEL[dataset]] + cells)

    if rows:
        tables.show(f"{name}: miary poboczne, wielkosci efektu",
                    header, rows,
                    note=f"Wartosc wariantu, w nawiasie roznica wobec "
                         f"{reference_label}; wiersz {reference_label} podaje sama "
                         f"wartosc. Wartosci to poziomy, wiec srednie po zapytaniach; "
                         f"roznica jest roznica miedzy konfiguracjami, wiec srednia "
                         f"roznic po odcinkach - i dlatego nie jest roznica dwoch "
                         f"wypisanych poziomow. Bez przedzialow ufnosci i bez "
                         f"orzekania o istotnosci - wnioskowanie prowadzone jest "
                         f"wylacznie dla {METRIC}.")

## 7. Kolekcje po zamrożeniu strategii

Liczebność przeszukiwanych kolekcji i średnia długość fragmentu dla strategii wskazanej niżej.

**Segmentacja części testowej.** Bufor segmentów nie zna podziału - jeden plik na parę (strategia, zbiór), z kolumną `split` - więc `run_segmentation.py` nie ma flagi `--split`, a jedno uruchomienie pokrywa dev i test naraz:

```powershell
python scripts/run_segmentation.py configs/e1b_office.yaml configs/e1b_tbbt.yaml
```

Konfiguracje `e1b_*` niosą zwycięską strategię `shots_histogram`. Oba pliki w jednym wywołaniu, bo skrypt grupuje je po zbiorze i dzieli jeden przebieg dekodowania na odcinek między strategie tego samego zbioru. Przebieg jest wznawialny: odcinek już policzony jest pomijany, `--force` przelicza go od nowa.

Odcinek wchodzi do segmentacji dopiero wtedy, gdy stoi w `<zbiór>_ranges.csv`, a tam trafia po oznaczeniu masek. Dopóki oba pliki zakresów zawierają wyłącznie odcinki deweloperskie, powyższe polecenie nie policzy nic nowego - najpierw maski odcinków testowych (`docs/03_warstwa_adnotacji.md`).

In [ ]:
WINNER_E1 = "shots_histogram"   # <- the E1 winner, as named in the configuration

rows = []
for dataset in DATASETS:
    path = datasets.segments_csv(WINNER_E1, dataset)
    fragments = [r for r in seg.load(path) if r["split"] == SPLIT] if path.exists() else []
    if not fragments:
        rows.append([LABEL[dataset], "-", "-"])
        continue
    lengths = [r["duration"] for r in fragments]
    rows.append([LABEL[dataset], len(fragments),
                 tables.number(sum(lengths) / len(lengths), 1)])

tables.show(f"Kolekcje zbioru deweloperskiego, strategia {WINNER_E1} "
            f"({STRATEGIES[WINNER_E1]})",
            ["Zbior", "Fragmentow", "Sr. dlugosc [s]"], rows)

## 8. E2 - wybór reprezentacji bazowej

Decyzję rozstrzyga porównanie E2-A z E2-B. Dwa warianty kontrolne są w tabeli, ale nie uczestniczą w decyzji: E2-Ap (w pracy E2-A') oddziela wpływ rozmiaru enkodera od wpływu typu reprezentacji, E2-Bp (w pracy E2-B') odejmuje mechanizm podpowiedzi, żeby oddzielić go od samej reprezentacji.

In [ ]:
group, columns = available(E2)
if not missing(E2) and group:
    reference = group[REFERENCE_E2]
    results = {label: compare.compare_variant(runs_of, reference, METRIC)
               for label, runs_of in group.items() if label != REFERENCE_E2}

    header = ["Konfiguracja"]
    for dataset in columns:
        header += [f"{LABEL[dataset]} R@10", f"{LABEL[dataset]} delta [95% PU]"]
    header += ["Polaczone delta [95% PU]"]

    rows = []
    for label in E2:
        if label not in group:
            continue
        cells = [E2_NAMES[label]]
        for dataset in columns:
            if label == REFERENCE_E2:
                cells += [tables.percent(compare.query_mean(
                    reference[dataset]["per_query"], METRIC)), "-"]
            else:
                entry = results[label]["per_dataset"][dataset]
                cells += [tables.percent(entry["value"]), tables.interval(entry)]
        cells += ["-" if label == REFERENCE_E2
                  else tables.interval(results[label]["pooled"])]
        rows.append(cells)

    tables.show("E2: reprezentacje bazowe", header, rows)

    deciding = {l: r for l, r in results.items() if l in E2_DECISION}
    if deciding:
        verdict = compare.decide(deciding, [l for l in E2_DECISION if l in group])
        print(f"\nwerdykt (bez wariantu kontrolnego): {verdict['winner']} -"
              f" {verdict['reason']}")
        if verdict["provisional"]:
            print(f"UWAGA: przedzial polaczony opiera sie na {verdict['series']} serialu")

## 9. E2 - rozkład efektu i narzut wariantu wideo-językowego

Każda kolumna to jeden krok, w którym zmienia się dokładnie jedna rzecz: rozmiar enkodera (Ap-A), typ reprezentacji wraz z danymi douczania (Bp-Ap), mechanizm dopasowania podpowiedzi (B-Bp). Ostatnia kolumna to narzut czasowy wariantu B; rośnie z liczbą fragmentów, bo dopasowanie tekst-wideo zależy w nim od cech ocenianego fragmentu.

In [ ]:
group, columns = available(E2)
if group and set(E2) <= set(group):
    size = compare.compare_variant(group["E2-Ap"], group["E2-A"], METRIC)
    kind = compare.compare_variant(group["E2-Bp"], group["E2-Ap"], METRIC)
    prompt = compare.compare_variant(group["E2-B"], group["E2-Bp"], METRIC)
    both = compare.compare_variant(group["E2-B"], group["E2-A"], METRIC)

    def median_ms(label, dataset):
        return (group[label][dataset]["metrics"].get("query_time_ms") or {}).get("median")

    rows = []
    for dataset in columns:
        base = median_ms("E2-A", dataset)
        prompted = median_ms("E2-B", dataset)
        overhead = prompted - base if (base and prompted) else None
        rows.append([LABEL[dataset],
                     tables.points(size["per_dataset"][dataset]["mean"]),
                     tables.points(kind["per_dataset"][dataset]["mean"]),
                     tables.points(prompt["per_dataset"][dataset]["mean"]),
                     tables.points(both["per_dataset"][dataset]["mean"]),
                     tables.number(median_ms("E2-Bp", dataset), 1),
                     tables.number(prompted, 1),
                     tables.number(overhead, 1) if overhead is not None else "-"])

    tables.show("E2: rozklad efektu i narzut",
                ["Zbior", "rozmiar (Ap-A)", "typ i dane (Bp-Ap)", "podpowiedzi (B-Bp)",
                 "laczny (B-A)", "Bp [ms/zap.]", "B [ms/zap.]", "narzut B [ms/zap.]"],
                rows,
                note="Narzut to roznica median czasu obslugi zapytania miedzy wariantem "
                     "z podpowiedziami (B) a wariantem A, na tej samej kolekcji.")
else:
    missing(E2)

## 10. E2 - rozbicie eksploracyjne po znaczniku `wymaga_ruchu`

Różnica B-Ap osobno na zapytaniach o ruch i na pozostałych. Rozbicie nie uczestniczy w decyzji: podawane jest jako wielkość efektu, bez orzekania o istotności, bo podzbiór jest na zbiorze deweloperskim mały.

In [ ]:
CONTRAST_E2 = "requirements:wymaga_ruchu"

group, columns = available(["E2-Ap", "E2-B"])
if group and len(group) == 2:
    matching, rest, empty = {}, {}, []
    for dataset in columns:
        matching[dataset], rest[dataset] = compare.subset_ids(dataset, SPLIT, CONTRAST_E2)
        if not matching[dataset]:
            empty.append(dataset)
    if empty:
        print(f"brak znacznikow w: {', '.join(LABEL[d] for d in empty)}"
              " - uzupelnij <zbior>_query_tags.csv")
    else:
        contrast = compare.contrast_variant(group["E2-B"], group["E2-Ap"], METRIC,
                                            matching, rest)
        rows = [[LABEL[dataset],
                 len(matching[dataset]),
                 tables.points(contrast["per_dataset"][dataset]["matching_delta"]),
                 tables.points(contrast["per_dataset"][dataset]["rest_delta"])]
                for dataset in columns]
        pooled_matching = {d: matching[d] for d in columns}
        rows.append(["polaczone odcinki", sum(len(v) for v in pooled_matching.values()),
                     tables.points(contrast["pooled"]["mean"]), "-"])
        tables.show("E2: roznica B-Ap wedlug znacznika wymaga_ruchu",
                    ["Zbior", "Zapytan ze znacznikiem", "delta ze znacznikiem",
                     "delta bez znacznika"], rows,
                    note="Wielkosci efektu, bez przedzialow - rozbicie nie uczestniczy "
                         "w decyzji.")
else:
    missing(["E2-Ap", "E2-B"])

## 11. E3-E5 - wybory wariantowe

Każdy wiersz to porównanie dwóch wariantów tego samego sygnału na samej BAZIE. Wartość dodatnia wskazuje wariant pierwszy, ten prostszy według hierarchii złożoności z rozdziału 4, który wygrywa również wtedy, gdy przedział obejmuje zero.

In [ ]:
header = ["Wariant"] + [f"{LABEL[d]} {METRIC}" for d in DATASETS]     + ["Polaczone delta [95% PU]", "Decyzja"]
rows, gaps = [], []
for simpler, complex_, name in PAIRS:
    result, verdict = pair(simpler, complex_)
    if result is None:
        gaps += [l for l in (simpler, complex_) if l not in runs]
        rows.append(name)
        rows.append([f"{simpler} / {complex_}"] + ["-"] * (len(DATASETS) + 2))
        continue

    rows.append(name)                       # a plain string becomes a subheading
    # the reference row first: its value travels inside the comparison, as
    # `reference_value`, so both variants come from one source
    rows.append([simpler] + [tables.percent(result["per_dataset"][d]["reference_value"])
                             for d in DATASETS] + ["-", ""])
    rows.append([complex_] + [tables.percent(result["per_dataset"][d]["value"])
                              for d in DATASETS]
                + [tables.interval(result["pooled"]),
                   verdict["winner"] + (" (prowizorycznie)" if verdict["provisional"]
                                        else "")])

tables.show("Decyzje wariantowe E3-E5", header, rows,
            align="l" + "r" * (len(header) - 2) + "l",
            note="W kazdej parze pierwszy wiersz to wariant prostszy, on jest "
                 "odniesieniem. Roznica liczona jako zlozony minus prostszy. Przy "
                 "przedziale obejmujacym zero wygrywa prostszy - regula z rozdzialu 4.")

# E5-Bp is a CONTROL, not a candidate: the same mechanism as E5-B asked with the
# whole query instead of its expression phrases. The gap between the two rows is
# the price of restricting the query to phrases, with everything else held. It
# decides nothing, so it stands apart from the pairs above -- and it behaves like
# the rest of the notebook when its runs are not there yet.
CONTROL = "E5-Bp"
if not missing([CONTROL, "E5-B"]):
    group, columns = available([CONTROL, "E5-B"])
    if group:
        control_rows = [
            ["E5-B: frazy mimiczne"] + [tables.percent(compare.query_mean(
                group["E5-B"][d]["per_query"], METRIC)) for d in columns],
            [f"{CONTROL}: cale zapytanie"] + [tables.percent(compare.query_mean(
                group[CONTROL][d]["per_query"], METRIC)) for d in columns],
        ]
        tables.show("Kontrola E5-Bp: fraza wobec calego zapytania",
                    ["Wariant"] + [f"{LABEL[d]} {METRIC}" for d in columns],
                    control_rows, align="l" + "r" * len(columns),
                    note="Wiersz kontrolny, nie kandydat: oba warianty maja ten sam "
                         "mechanizm regionow twarzy i roznia sie wylacznie tym, czy "
                         "pytaja slownik fraza mimiczna, czy calym zdaniem zapytania.")

if gaps:
    print(f"\nbrak przebiegow: {', '.join(sorted(set(gaps)))}")

## 12. Przejścia przy decyzjach fazy deweloperskiej

Każda decyzja tej fazy jest porównaniem przebiegów, a różnica średnich mówi tylko, o ile. Ta tabela mówi, kogo: ile zapytań wariant naprawił ($0\to1$), ile zepsuł ($1\to0$) i ile zostawił bez zmiany.

Trzy bloki, każdy z własnym odniesieniem: E1 wobec stałych okien, E2 wobec OpenCLIP ViT-H/14, a E3-E5 wobec BAZY. W bloku E3-E5 stoi więc każdy wariant, zamrożony i odrzucony, oraz wariant kontrolny E5-Bp; o decyzji rozstrzyga para wierszy tego samego sygnału.

Kolumna "zmienionych" to suma dwóch pierwszych. Kolumna "blisko progu" podaje, ile z nich zmieniło status wyłącznie dlatego, że próg miary wypadł między dwoma niemal identycznymi rankingami - poprawny fragment leży wtedy w obu przebiegach na randze 7-14 (`compare.BORDERLINE_BAND`).

Różnica dwóch pierwszych kolumn podzielona przez `n` daje wkład liczony po zapytaniach.

In [ ]:
# E3-E5: every variant that puts one component on the base, in pair order, with
# the E5 control beside its pair. The same layout as tab:app-przejscia on the test
# part -- a pair reported as one row would show only one of its two variants.
VARIANTS = [label for simpler, complex_, _ in PAIRS for label in (simpler, complex_)]
VARIANTS.insert(VARIANTS.index("E5-B") + 1, "E5-Bp")


def named(label):
    """Row label of the E3-E5 block: the experiment and the component it adds."""
    return f"{label}: BAZA + {exp.LABEL_COMPONENT[label]}"


# (candidate, reference, block, row label)
DECISIONS = ([(label, REFERENCE_E1, "E1", exp.display(label))
              for label in E1 if label != REFERENCE_E1]
             + [(label, REFERENCE_E2, "E2", exp.display(label))
                for label in E2 if label != REFERENCE_E2]
             + [(label, exp.BASE, "E3-E5", named(label)) for label in VARIANTS])

rows = []
for candidate, reference, block, described in DECISIONS:
    group, columns = available([candidate, reference])
    if len(group) < 2:
        continue
    for dataset in columns:
        after, before = group[candidate][dataset], group[reference][dataset]
        counts = compare.transitions(after, before)
        total = sum(counts.values())
        rows.append([block, described, LABEL[dataset], total,
                     counts["0->1"], counts["1->0"], counts["1->1"], counts["0->0"],
                     counts["0->1"] + counts["1->0"], compare.borderline(after, before),
                     tables.points((counts["0->1"] - counts["1->0"]) / total
                                   if total else None)])

if rows:
    tables.show("Przejscia przy decyzjach fazy deweloperskiej",
                ["Blok", "Porownanie", "Zbior", "n", "0->1", "1->0", "1->1", "0->0",
                 "Zmienionych", "Blisko progu", "Delta [p.p.]"], rows,
                align="lll" + "r" * 8,
                note="Odniesienie: E1 - stale okna (E1-A), E2 - OpenCLIP ViT-H/14 "
                     "(E2-A), E3-E5 - BAZA. Cztery grupy sa rozlaczne i wyczerpujace, "
                     "wiec ich suma jest kolumna n. Blisko progu: ranga poprawnego "
                     f"fragmentu w przedziale {compare.BORDERLINE_BAND[0]}-"
                     f"{compare.BORDERLINE_BAND[1]} w obu przebiegach naraz.")
else:
    print("brak par przebiegow do porownania")

## 13. Zamrożona konfiguracja

Zestawienie wszystkich pięciu rozstrzygnięć razem z podstawą każdego z nich. Wiersze, których nie da się jeszcze rozstrzygnąć, wypisują `?`.

In [ ]:
def resolve(labels, reference, decision=None):
    """Werdykt reguly decyzji dla podanej rodziny wariantow, albo '?'."""
    group, _ = available(labels)
    considered = decision or labels
    if len(group) < 2 or reference not in group:
        return "?", ""
    results = {label: compare.compare_variant(group[label], group[reference], METRIC)
               for label in considered if label in group and label != reference}
    if not results:
        return "?", ""
    verdict = compare.decide(results, [l for l in considered if l in group])
    return verdict["winner"], (" (prowizorycznie)" if verdict["provisional"] else "")


e1, e1_note = resolve(E1, REFERENCE_E1)
e2, e2_note = resolve(E2, REFERENCE_E2, E2_DECISION)
rows = [
    ["strategia segmentacji (BAZA)", f"{e1}{e1_note}", "E1"],
    ["reprezentacja bazowa (BAZA)", f"{e2}{e2_note}", "E2"],
]
for simpler, complex_, name in PAIRS:
    _, verdict = pair(simpler, complex_)
    winner = verdict["winner"] if verdict else "?"
    note = " (prowizorycznie)" if verdict and verdict["provisional"] else ""
    element = {"E3-B": "generator opisow (full)", "E4-B": "detektor obiektow (full)",
               "E5-B": "mechanizm mimiki (full)"}[simpler]
    rows.append([element, f"{winner}{note}", simpler[:2]])

tables.show("Zamrozona konfiguracja po fazie deweloperskiej",
            ["Element", "Rozstrzygniecie", "Podstawa"], rows, align="lll",
            note="Sygnal tozsamosci wchodzi do potoku PELNEGO bezwarunkowo (tylko material "
                 "serialowy), a wariant YOLOE sterowany zapytaniem pozostaje poza nim - "
                 "obie decyzje sa niezalezne od wynikow i nie sa tu rozstrzygane.")